# 05 · Sanitization: redact, label, number, tokenize

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/05_sanitization.ipynb)

**Runtime:** CPU is fine.

← [04_composite_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) · [series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md) · [06_search_on_sanitized_data](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb) →

Detection tells you *where* the sensitive data is. **Sanitization** decides
what goes in its place, and that choice determines what the text is still good
for. This notebook renders four modes side by side, using the predictions
saved in 02.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"
USE_DRIVE = False      # True: keep results in Google Drive so the other notebooks reuse them

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup(drive=USE_DRIVE)

## Four modes

| mode | looks like | what it preserves |
|---|---|---|
| `redact` | `********` | Nothing. Fixed-length asterisks, so not even the length leaks. |
| `label` | `[EMAIL]` | The category only. Every email looks the same. |
| `label_number` | `[EMAIL_1]` | Identity **within one document**: the same value gets the same number, and different values get different numbers. Numbering restarts in every document. |
| `label_token` | `[EMAIL_jRc7QGn]` | Identity **across documents and over time**. A token vault maps each value to a deterministic token: the same plaintext always gets the same token, whichever detector found it. |

`label` is the default in the API's `/v1/sanitize` endpoint. `label_token` is
the one that keeps text joinable and searchable, as
[06](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb) shows.

In [ ]:
from opf_eval import scoring
from opf_eval.demo import sanitize

S = nb.load_session()
run_dir = nb.ensure_run(S)
run = scoring.Run.load(run_dir)

# === EDIT ME ===
N_SHOW = 6
DETECTORS = [d for d in S.detectors if d in run.detectors]
MODES = ["redact", "label", "label_number", "label_token"]
# ================

## Token vault (optional)

`label_token` needs a Skyflow vault that issues deterministic tokens. The
one-time setup is in
[docs/token-vault-setup.md](https://github.com/jstjoe/local-privacy/blob/main/docs/token-vault-setup.md).
It needs these secrets:

- `SKYFLOW_TOKEN_VAULT_URL`
- `SKYFLOW_TOKEN_VAULT_ID`
- `SKYFLOW_TOKEN_BEARER_TOKEN`, or `SKYFLOW_BEARER_TOKEN` if you already set
  it in 02

Without them, the `label_token` column shows a placeholder.

In [ ]:
TOKEN_VAULT = nb.token_vault()
print("token vault:", "configured" if TOKEN_VAULT else "not configured — label_token will show a placeholder")

## Side by side

One table per record. Rows are detectors and columns are modes. Read across a
row to see what each mode does with the same spans. Read down a column to see
how detectors disagree: a missed span leaks in *every* mode.

In [ ]:
for ex in run.fixtures[:N_SHOW]:
    preds = {d: run.predictions[d][ex["id"]]["spans"] for d in DETECTORS if ex["id"] in run.predictions[d]}
    nb.show_md(sanitize.side_by_side(ex, preds, modes=MODES, token_vault=TOKEN_VAULT))

**Reading the rows**

- If `label` and `label_number` look identical, the detector found at most one
  value per category in that record.
- A value that appears unchanged in every mode is a **miss**. Sanitization can
  only be as good as detection.
- Overlapping spans are resolved before rendering: the earlier span wins and
  the overlap is left untouched.

## Your own text

Sanitize any string with any detector. This builds the detector live, so the
first call includes loading the model.

In [ ]:
from opf_eval.detectors import registry
from opf_eval.transforms import render_modes

# === EDIT ME ===
TEXT = "Ping Priya Raman (priya.raman@example.com, +44 20 7946 0958) about invoice 4471 — she'll cc priya.raman@example.com."
DETECTOR = DETECTORS[0] if DETECTORS else "presidio"
# ================
det = registry.build(DETECTOR, device=S.device or nb.device())
spans = det.detect(TEXT)["spans"]
for mode, out in render_modes(TEXT, spans, modes=MODES, token_vault_client=TOKEN_VAULT).items():
    print(f"{mode:>13}  {out}")

## Next

See what each mode does to a real downstream task: search.

**[06_search_on_sanitized_data](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb)** — Use: search over sanitized text.